<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/TwitterSentimentAnalaysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kaggle**

In [ ]:
# Core dependencies, if needed: %pip install pandas numpy matplotlib scikit-learn joblib
# Optional download dependency: %pip install kagglehub
# No Kaggle CLI or NLTK corpus installation is needed for the model pipeline.

In [ ]:
# Locate the explicit Sentiment140 training CSV; do not choose an arbitrary file.
from pathlib import Path
DATA_PATH = Path('training.1600000.processed.noemoticon.csv')
if not DATA_PATH.exists():
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError('Place the Sentiment140 training CSV beside this notebook '
                          'or install kagglehub.') from error
    path = Path(kagglehub.dataset_download('kazanova/sentiment140'))
    DATA_PATH = path / 'training.1600000.processed.noemoticon.csv'

# **Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, log_loss, roc_auc_score)
import joblib
import sklearn
RANDOM_STATE = 42

In [ ]:
# Stopword removal is optional, not a required NLP step.
# Keep words such as "not", "no", and "never": they can change sentiment.
# Using the vectorizer directly avoids repeated NLTK stopword lookups per token.

In [ ]:
# No external stopword download is needed. Examples worth preserving:
print(['not good', 'never again', "isn't bad", 'no thanks'])

# **Reading & Exploring the Data**

In [ ]:
# Reading the path
print('Dataset:', DATA_PATH)

In [ ]:
# Converting to pandas df: this CSV has no header and uses Latin-1 encoding.
col_names = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv(DATA_PATH, names=col_names, encoding='ISO-8859-1',
                 dtype={'id': 'string', 'text': 'string', 'user': 'string'})
df.head(5)

In [ ]:
df.info()
# info() shows missing values but does not establish that there are no duplicates.
print('Missing values:')
print(df.isna().sum())
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Repeated tweet IDs:', int(df['id'].duplicated().sum()))
print('Repeated raw texts:', int(df['text'].duplicated().sum()))

In [ ]:
# Replacing 4 with 1: 0 --> Negative, 4 --> Positive.
# This is a convenient explicit label convention; sklearn also accepts other labels.
if df['target'].isna().any() or not df['target'].isin([0, 4]).all():
    raise ValueError('Expected only labels 0 and 4 in the Sentiment140 training file.')
print('Before:', df['target'].value_counts().to_dict())
df['target'] = df['target'].map({0: 0, 4: 1}).astype(int)
print('After:', df['target'].value_counts().to_dict())

# Text preprocessing with TF-IDF
Stemming is an optional experiment, not a requirement. The baseline now uses lowercase words, contractions, punctuation tokens, and word bigrams. It keeps stopwords so phrases such as "not good" survive. TF-IDF produces sparse numerical features, rather than semantic neural embeddings.

Original stemming explanations are retained in the appendix. In Porter stemming, different forms are not guaranteed to share a root: for example, "runner" need not become "run".

In [ ]:
# No stemmer is needed for this baseline.
# TfidfVectorizer performs tokenization/lowercasing inside the saved pipeline.

In [ ]:
# Stateless key for detecting repeated text, not a learned text representation.
# Lowercasing and whitespace normalization conservatively identify duplicate tweets.
# We retain raw text for the model, including contractions and punctuation.
def normalized_text_keys(texts):
    return texts.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
# Clean rows jointly before splitting to preserve text/label alignment.
before = len(df)
df = df.dropna(subset=['id', 'text']).copy()
df['text_key'] = normalized_text_keys(df['text'])
df = df.loc[df['text_key'].ne('')].copy()
# Do not arbitrarily pick a label for conflicting copies of a tweet ID or text.
for column in ['id', 'text_key']:
    label_counts = df.groupby(column)['target'].nunique()
    conflicting_keys = label_counts.index[label_counts.gt(1)]
    conflict_mask = df[column].isin(conflicting_keys)
    print(f'{column}: rows excluded for conflicting labels:', int(conflict_mask.sum()))
    df = df.loc[~conflict_mask].drop_duplicates(subset=column).copy()
print('Total missing/blank/conflicting/repeated rows removed:', before - len(df))
# Deduplication changes the evaluation population to distinct non-conflicting texts.
# Near-duplicates, retweets with edits, and repeated users need additional audits.

In [ ]:
df[['text', 'target']].head(10)

# **Separating the Data**

In [ ]:
# Only tweet text is a feature: exclude target, tweet ID, date, flag, and username fields.
x = df['text'].astype(str)
y = df['target']
# No stemming or corpus-learned transformation is fitted before splitting.

In [ ]:
# Stratified 80/10/10 train/development/test split on deduplicated texts.
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
x_dev, x_test, y_dev, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)
for name, texts, labels in [('train', x_train, y_train),
                             ('development', x_dev, y_dev), ('test', x_test, y_test)]:
    assert texts.index.equals(labels.index)
    if labels.nunique() != 2:
        raise ValueError(f'{name} must contain both classes.')
    print(name, len(texts), labels.value_counts(normalize=True).to_dict())
# Verify normalized text keys are disjoint across splits.
train_keys = set(df.loc[x_train.index, 'text_key'])
dev_keys = set(df.loc[x_dev.index, 'text_key'])
test_keys = set(df.loc[x_test.index, 'text_key'])
assert train_keys.isdisjoint(dev_keys) and train_keys.isdisjoint(test_keys)
assert dev_keys.isdisjoint(test_keys)
del train_keys, dev_keys, test_keys

In [ ]:
# Creating TF-IDF features inside a complete raw-text-to-prediction pipeline.
# Vocabulary and inverse document frequency are learned from TRAINING text only.
# Word bigrams represent phrases such as "not good"; stopwords are retained.
vectorizer = TfidfVectorizer(
    lowercase=True, stop_words=None, ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b|[!?]+",
    min_df=2, max_features=100_000, sublinear_tf=True, dtype=np.float32,
)
# min_df/max_features bound vocabulary growth for the 1.6-million-row dataset.
# The sparse matrix stays sparse; never call toarray() on the full corpus.
# TF measures frequency within a document; IDF downweights broadly common terms.
# Calling fit_transform on test text would relearn vocabulary/IDF and leak information.

In [ ]:
# Keep x_train/x_dev/x_test as raw text for a simpler training/inference interface.
print('Example raw training tweet:', x_train.iloc[0])

# **Creating & Training the Model**

In [ ]:
Logistic_model = Pipeline([
    ('tfidf', vectorizer),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear',
                                      random_state=RANDOM_STATE)),
])
Logistic_model.fit(x_train, y_train)
print('Training completed; vocabulary size:', len(Logistic_model.named_steps['tfidf'].vocabulary_))
# No balancing is assumed necessary. Check training class proportions first.

In [ ]:
# Development evaluation guides future choices without touching the test set.
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(x_train.to_frame(), y_train)
dev_predictions = Logistic_model.predict(x_dev)
print(f'Development accuracy: {accuracy_score(y_dev, dev_predictions):.4f}')
print(f'Development macro F1: {f1_score(y_dev, dev_predictions, average="macro"):.4f}')
print(f'Majority baseline development accuracy: {baseline.score(x_dev.to_frame(), y_dev):.4f}')
print(f'Majority baseline development macro F1: '
      f'{f1_score(y_dev, baseline.predict(x_dev.to_frame()), average="macro"):.4f}')
# .score() returns accuracy for these classifiers. Training accuracy alone is insufficient.
# Tune settings here if needed, then finalize before running the next cell.

In [ ]:
# Final test evaluation: run once after preprocessing/model choices are complete.
positive_index = list(Logistic_model.classes_).index(1)
y_pred_proba = Logistic_model.predict_proba(x_test)[:, positive_index]
y_pred_int = Logistic_model.predict(x_test)  # Hard labels, not probabilities
accuracy = accuracy_score(y_test, y_pred_int)
loss = log_loss(y_test, np.column_stack([1 - y_pred_proba, y_pred_proba]), labels=[0, 1])
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test macro F1: {f1_score(y_test, y_pred_int, average="macro"):.4f}')
print(f'Log Loss: {loss:.4f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')
print(classification_report(y_test, y_pred_int, labels=[0, 1],
      target_names=['Negative', 'Positive'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_int, labels=[0, 1],
    display_labels=['Negative', 'Positive'], cmap='Blues')
plt.show()
# Log loss needs probabilities. A constant class-prior predictor is a useful reference:
prior = float(y_train.mean())
prior_proba = np.tile([1 - prior, prior], (len(y_test), 1))
print(f'Class-prior reference log loss: {log_loss(y_test, prior_proba, labels=[0, 1]):.4f}')

In [ ]:
print(y_pred_proba[:10])  # Probability estimates for the positive label

In [ ]:
print(y_pred_int[:10])  # Hard class predictions

# **Saving the Trained Model**

In [ ]:
# joblib saves the full sklearn pipeline, including vocabulary, IDF, and classifier.
# Only load artifacts you trust, using matching library versions.
MODEL_PATH = Path('twitter_sentiment_pipeline.joblib')

In [ ]:
# Saving the complete trained model enables predictions from RAW tweets later.
artifact = {
    'pipeline': Logistic_model,
    'labels': {0: 'Negative Tweet', 1: 'Positive Tweet'},
    'sklearn_version': sklearn.__version__,
    'numpy_version': np.__version__,
    'training_rows': len(x_train),
}
joblib.dump(artifact, MODEL_PATH, compress=3)
print('Saved:', MODEL_PATH.resolve())
# Reuse, deployment, and sharing require the preprocessing as well as the classifier.
# The pipeline uses only library components, so no notebook-defined stemmer is required.

# **Using the Saved Model for Future Predictions**

In [ ]:
# Using the Saved Model for Future Predictions: portable relative path, not /content.
loaded_artifact = joblib.load(MODEL_PATH)
if loaded_artifact['sklearn_version'] != sklearn.__version__:
    raise RuntimeError('Use the same scikit-learn version as the saved artifact.')
loaded_model = loaded_artifact['pipeline']

In [ ]:
# Pass a list of raw strings, not an already-vectorized row.
x_new = ["I am not happy with this service!"]

In [ ]:
print('Input:', x_new[0])
# This is a new example, not a labeled test row; there is no known target to print.

In [ ]:
prediction = loaded_model.predict(x_new)
# Check that reloading preserves predictions on a few held-out raw tweets.
np.testing.assert_array_equal(loaded_model.predict(x_test.iloc[:10]),
                              Logistic_model.predict(x_test.iloc[:10]))

In [ ]:
print(prediction)

In [ ]:
print(loaded_artifact['labels'][int(prediction[0])])

In [ ]:
# End-to-end interface: loaded_model.predict([raw_tweet])
# The same lowercasing, tokenization, vocabulary, and IDF are applied automatically.

## Review notes and next experiments
- The old TF-IDF fitting was correct: fit on training text, transform test text. This behavior is now enforced by a Pipeline.
- The old log-loss cell referenced undefined `y_pred`; it now uses the actual probability estimates.
- Saving the classifier alone omitted the fitted vectorizer and relied on an external stemmer. The new artifact includes all preprocessing and accepts raw tweets in a fresh process.
- Stopword removal discarded negation, and alphabet-only cleaning damaged contractions. The baseline keeps both and adds bigrams. Stemming is an experiment, not automatically an improvement.
- IDs and normalized text are deduplicated before splitting; conflicting labels are excluded and counted. That changes the evaluation population. Compare against a grouped split if retaining repeated tweets is important. Exact-text checks do not eliminate paraphrase leakage.
- This random split estimates performance on similar tweets, not future tweets or unseen authors. Users may appear across splits, and username mentions/URL fragments remain in tweet text. For those deployment questions, use author-grouped or chronological evaluation and compare mention/URL normalization on development data.
- This is binary negative/positive classification: neutral, mixed sentiment, sarcasm, and newer language need separate evaluation. Classifier probabilities are not automatically calibrated confidence.
- Full training can require substantial RAM/time. The vocabulary cap bounds columns, not total sparse-matrix size. For quick experiments, sample a stratified subset before creating the three splits and report that changed scope.
- Keep the trained model fixed after final evaluation. A later refit should be versioned as a new artifact with its own validation.

References: [scikit-learn text features](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), [model persistence](https://scikit-learn.org/stable/model_persistence).

## Original comments and learning notes
Changed original cells are preserved below for reference. Executable cells above contain the corrections.

<details>
<summary>Original cell 3</summary>

```text
!pip install kaggle
```

</details>

<details>
<summary>Original cell 4</summary>

```text
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")
```

</details>

<details>
<summary>Original cell 6</summary>

```text
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re # provides regular expression operations

from nltk.corpus import stopwords
"""imports a list of common 'stop words' (e.g., 'the', 'is', 'and') from the
Natural Language Toolkit (NLTK) library,which are often removed from text data before analysis."""

from nltk.stem.porter import PorterStemmer
"""
used to reduce words to their base or root form (e.g., 'running' to 'run').
"""

from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF is a common way to represent text data numerically for machine learning models.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
```

</details>

<details>
<summary>Original cell 7</summary>

```text
import nltk
nltk.download('stopwords') # Downloading the Stopwords Corpus
# Stopwords are often removed from the text before analysis because they typically don't carry significant meaning.
```

</details>

<details>
<summary>Original cell 8</summary>

```text
print(stopwords.words('english'))
```

</details>

<details>
<summary>Original cell 10</summary>

```text
#@title Reading the path
file_list = os.listdir(path)
print(file_list)
```

</details>

<details>
<summary>Original cell 11</summary>

```text
#@title Converting to pandas df
col_names = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv(os.path.join(path, file_list[0]), names=col_names, encoding='ISO-8859-1') # constructs the full path to a CSV file by joining the path and the first element of file_list.
df.head(5)
```

</details>

<details>
<summary>Original cell 12</summary>

```text
df.info()

"""
No missing values were found
No Duplicates were found
"""
```

</details>

<details>
<summary>Original cell 13</summary>

```text
#@title Replacing 4 with 1
"""
0 --> Negative
4 --> Positive

The conversion makes the data more compatible with standard
classification algorithms that expect binary labels (0 or 1).
"""

print(df['target'].value_counts()) # Before
df.replace({'target':{4:1}}, inplace=True)
print(df['target'].value_counts()) # After
#
```

</details>

<details>
<summary>Original cell 14</summary>

```text
# **Stemming: Pre-processing step**

Is a process of reducing the word to its root word. For example, 'running', 'runs', and 'runner' would all be reduced to 'run'.
```

</details>

<details>
<summary>Original cell 15</summary>

```text
port_stem = PorterStemmer() # PorterStemmer is a widely used algorithm in NLP for 'stemming' words.
```

</details>

<details>
<summary>Original cell 16</summary>

```text
def stemming(content):
  """
  This function will be applied to each sample individually
  """
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content) #  remove all characters that are not English alphabet letters (symbols, punctuations, marks)
  stemmed_content = stemmed_content.lower() # Ensures that each letter treated the same to avoid case-sensitivity
  stemmed_content = stemmed_content.split() #  A basic form of tokenization, breaks the cleaned string into a list of words

  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  """
  Stemming is applied to all words EXCEPT stopwords since they do not carry
  meaningful information for sentiment analysis.
  """
  stemmed_content = ' '.join(stemmed_content) # joins the processed list of stemmed words back into a single string, with each word separated by a space
  return stemmed_content
```

</details>

<details>
<summary>Original cell 17</summary>

```text
df['stemmed_text'] = df['text'].apply(stemming)
```

</details>

<details>
<summary>Original cell 18</summary>

```text
df['stemmed_text'].head(10)
```

</details>

<details>
<summary>Original cell 20</summary>

```text
x = df['stemmed_text']
y = df['target']
```

</details>

<details>
<summary>Original cell 21</summary>

```text
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    stratify=y,
                                                    random_state=2)
```

</details>

<details>
<summary>Original cell 22</summary>

```text
#@title Creating Embeddings using TF-IDF

vectorizer = TfidfVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)



"""
When you fit_transform on the test set, the vectorizer re-learns the
vocabulary and IDF values from the test set, which can lead to data leakage
and an inaccurate evaluation of your model.


Term Frequency (TF) measures how frequently a term appears in a document,
while Inverse Document Frequency (IDF) measures how important
a term is across the entire corpus.

Words that are common across many documents (like 'the', 'a') get a lower IDF
score, while unique or specific words get a higher score.
"""
```

</details>

<details>
<summary>Original cell 23</summary>

```text
print(x_train)
```

</details>

<details>
<summary>Original cell 25</summary>

```text
Logistic_model = LogisticRegression(max_iter=1000)
Logistic_model.fit(x_train, y_train)
```

</details>

<details>
<summary>Original cell 26</summary>

```text

train_accuracy = Logistic_model.score(x_train, y_train)
test_accuracy = Logistic_model.score(x_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

"""
The .score() method for scikit-learn classifiers typically returns the mean accuracy on the given test data and labels.
"""

# The Test Accuracy is already calculated in the next cell.
```

</details>

<details>
<summary>Original cell 27</summary>

```text
 from sklearn.metrics import log_loss

# For computing loss: Returns the probability estimates for each class for every sample in x_test
y_pred_proba = Logistic_model.predict_proba(x_test)[:, 1] # [:, 1] returns the probabilitis for the positive class

# For computing accuracy: Stores the hard class predictions (0 or 1), [Not the probabilities]
y_pred_int = Logistic_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred_int)
print(f"Test Accuracy: {accuracy:.4f}")

loss = log_loss(y_test, y_pred)
print(f"Log Loss: {loss:.4f}")

"""
A log loss of {0.47} falls between perfect and random, indicating that the
model is performing better than random chance.
"""
```

</details>

<details>
<summary>Original cell 28</summary>

```text
print(y_pred_proba)
```

</details>

<details>
<summary>Original cell 29</summary>

```text
print(y_pred_int)
```

</details>

<details>
<summary>Original cell 31</summary>

```text
import pickle
```

</details>

<details>
<summary>Original cell 32</summary>

```text
filename = "trained_model.sav" # Defines the file name the model will saved in.
pickle.dump(Logistic_model, open(filename, 'wb'))


"""
Why is this important? Saving your trained model allows you to:

Reuse it later: You don't have to retrain the model every time you want to use it for predictions on new data.
Deploy it: You can load the saved model into another application or environment to make predictions.
Share it: You can easily share your trained model with others without needing to share your entire training pipeline.

"""
```

</details>

<details>
<summary>Original cell 34</summary>

```text
loaded_model = pickle.load(open('/content/trained_model.sav', 'rb'))
```

</details>

<details>
<summary>Original cell 35</summary>

```text
x_new = x_test[24]
```

</details>

<details>
<summary>Original cell 36</summary>

```text
print(y_test.iloc[24])
```

</details>

<details>
<summary>Original cell 37</summary>

```text
prediction = loaded_model.predict(x_new)
```

</details>

<details>
<summary>Original cell 38</summary>

```text
print(prediction)
```

</details>

<details>
<summary>Original cell 39</summary>

```text
if prediction[0] == 0:
  print("Negative Tweet")
else:
  print("Positive Tweet")
```

</details>